# BEBM MLP: fixed h vs formula h vs gradient h

This notebook compares three BEBM MLP models trained on the same binary MNIST data:

- `BEBM MLP | fixed h`: visible field is initialized from data frequencies and kept fixed.
- `BEBM MLP | formula h`: visible field is recomputed from the MLP formula after each update.
- `BEBM MLP | gradient h`: visible field is a trainable parameter updated by SGD.

It follows the same analysis structure as `RBM_vs_BEBM_RBMEnergy_compare.ipynb`: spectra over training, data PCA basis, final permanent chains, generated samples over training, and mixing-time diagnostics.

Important: MLP checkpoints store weights but not the implementation rule used for the visible field. All three models use SiLU; the helper reloads checkpoints with the activation specified in `MODEL_SPECS`.

In [ ]:
from pathlib import Path
import contextlib

import h5py
import numpy as np
import torch
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from rbms.dataset import load_dataset
from rbms.io import load_model, load_params
from rbms.utils import get_saved_updates
from rbms.plot import plot_mult_PCA
import rbms.EBM_binary.energies as binary_energies

plt.rcParams["mathtext.fontset"] = "stix"
plt.rcParams["font.family"] = "STIXGeneral"
plt.rcParams.update({"font.size": 13})

%load_ext autoreload
%autoreload 2

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float32

train_dataset_name = "data/MNIST_train.h5"
test_dataset_name = "data/MNIST_test.h5"

MODEL_SPECS = {
    "BEBM MLP | fixed h": {
        "filename": "pcd_trains/BEBM_MLP_SiLU_h500_DMALA100_lr1e-2_100k.h5",
        "activation": torch.nn.SiLU,
    },
    "BEBM MLP | formula h": {
        "filename": "pcd_trains/BEBM_MLP_SiLU_formulaH_h500_DMALA100_lr1e-2_100k.h5",
        "activation": torch.nn.SiLU,
    },
    "BEBM MLP | gradient h": {
        "filename": "pcd_trains/BEBM_MLP_SiLU_gradH_h500_DMALA100_lr1e-2_100k.h5",
        "activation": torch.nn.SiLU,
    },
}

IMAGE_SHAPE = (28, 28)
PCA_POINT_SCALE = float(IMAGE_SHAPE[0])

print("device:", device)
for label, spec in MODEL_SPECS.items():
    print(label, "->", spec["filename"], "exists=" + str(Path(spec["filename"]).exists()))

## Load Data and Model Checkpoints

In [ ]:
train_dataset, test_dataset = load_dataset(
    dataset_name=train_dataset_name,
    test_dataset_name=test_dataset_name,
    device=device,
    dtype=dtype,
)

train_visible = train_dataset.data.to(device=device, dtype=dtype)
num_visibles = train_dataset.get_num_visibles()

saved_updates_by_model = {
    label: get_saved_updates(spec["filename"])
    for label, spec in MODEL_SPECS.items()
}

for label, updates in saved_updates_by_model.items():
    print(label)
    print("  first update:", int(updates[0]))
    print("  last update: ", int(updates[-1]))
    print("  checkpoints: ", len(updates))

## Helpers

In [ ]:
def selected_updates(updates, num_points):
    indices = np.linspace(0, len(updates) - 1, num_points).round().astype(int)
    indices = np.unique(indices)
    return np.asarray(updates)[indices]


def selected_updates_decades(updates):
    updates = np.asarray(updates)
    max_update = int(updates[-1])

    targets = [int(updates[0])]

    decade = 100
    while decade <= max_update:
        for multiplier in range(1, 10):
            target = multiplier * decade
            if target <= max_update:
                targets.append(target)
        decade *= 10

    targets.append(max_update)
    targets = np.unique(targets)

    selected = []
    for target in targets:
        index = np.argmin(np.abs(updates - target))
        selected.append(updates[index])

    return np.unique(selected)


class ActivationMLPEnergy(torch.nn.Module):
    def __init__(self, num_visibles, hidden_dim=256, num_layers=1, visible_field=None, activation=torch.nn.SiLU):
        super().__init__()
        self.num_visibles = num_visibles
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        if visible_field is None:
            visible_field = torch.zeros(num_visibles)
        self.register_buffer("visible_field", visible_field.clone())

        layers = []
        in_dim = num_visibles
        for _ in range(num_layers):
            layers.append(torch.nn.Linear(in_dim, hidden_dim))
            layers.append(activation())
            in_dim = hidden_dim
        layers.append(torch.nn.Linear(in_dim, 1))
        self.net = torch.nn.Sequential(*layers)

    def forward(self, v):
        return self.net(v).view(-1) - v @ self.visible_field


@contextlib.contextmanager
def mlp_activation_restore_context(activation):
    original_mlp = binary_energies.MLPEnergy
    original_map = binary_energies.ENERGY_MAP.get("mlp")

    def factory(num_visibles, hidden_dim=256, num_layers=1, visible_field=None):
        return ActivationMLPEnergy(
            num_visibles=num_visibles,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            visible_field=visible_field,
            activation=activation,
        )

    binary_energies.MLPEnergy = factory
    binary_energies.ENERGY_MAP["mlp"] = factory
    try:
        yield
    finally:
        binary_energies.MLPEnergy = original_mlp
        binary_energies.ENERGY_MAP["mlp"] = original_map


def load_params_with_activation(label, update):
    spec = MODEL_SPECS[label]
    with mlp_activation_restore_context(spec["activation"]):
        return load_params(spec["filename"], int(update), device=device, dtype=dtype)


def load_model_with_activation(label, update):
    spec = MODEL_SPECS[label]
    with mlp_activation_restore_context(spec["activation"]):
        return load_model(spec["filename"], int(update), device=device, dtype=dtype)


def read_mlp_weights_from_h5(filename, update):
    with h5py.File(filename, "r") as f:
        params_group = f[f"update_{int(update)}"]["params"]
        weight_keys = sorted(
            [key for key in params_group.keys() if key.startswith("net.") and key.endswith(".weight")],
            key=lambda key: int(key.split(".")[1]),
        )
        if len(weight_keys) < 2:
            raise KeyError(f"Expected at least two MLP weight tensors at update {update} in {filename}")
        weights = {key: torch.as_tensor(params_group[key][()], dtype=torch.float32) for key in weight_keys}
    return weights


def sample_visible(params, initial_visible, num_steps, alpha=0.5):
    chains = params.init_chains(num_samples=initial_visible.shape[0], start_v=initial_visible)
    chains = params.sample_state(
        chains=chains,
        n_steps=num_steps,
        kernel="dmala",
        kernel_params={"alpha": alpha},
    )
    acceptance = chains.get("acceptance")
    if torch.is_tensor(acceptance):
        acceptance = float(acceptance.detach().cpu())
    return chains["visible"].detach(), acceptance


def plot_binary_image_grid(visible, title, shape=(28, 28), grid_size=(8, 8)):
    visible = visible.detach().cpu()
    n_rows, n_cols = grid_size
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols, n_rows))
    for ax, img in zip(axes.ravel(), visible[: n_rows * n_cols]):
        ax.imshow(img.view(*shape), cmap="gray_r", vmin=0, vmax=1)
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

## Singular Values Over Training

For both models, `W1 = net.0.weight` has shape `(hidden_dim, num_visibles)` and `W2 = net.2.weight` has shape `(1, hidden_dim)`. We plot the top singular values for both MLP layers.

In [ ]:
SINGULAR_TOP_K = 10

singular_history = {}

for label, spec in MODEL_SPECS.items():
    updates = selected_updates_decades(saved_updates_by_model[label])
    w1_singular = []
    w2_singular = []

    for update in tqdm(updates, desc=f"Singular values: {label}"):
        weights = read_mlp_weights_from_h5(spec["filename"], update)
        weight_keys = sorted(weights, key=lambda key: int(key.split(".")[1]))
        W1 = weights[weight_keys[0]]
        W2 = weights[weight_keys[-1]]

        s1 = torch.linalg.svdvals(W1).sort(descending=True).values[:SINGULAR_TOP_K]
        s2 = torch.linalg.svdvals(W2).sort(descending=True).values[:SINGULAR_TOP_K]
        w1_singular.append(s1.cpu())
        w2_singular.append(s2.cpu())

    singular_history[label] = {
        "updates": updates,
        "W1_singular": torch.stack(w1_singular).numpy(),
        "W2_singular": torch.stack(w2_singular).numpy(),
    }

In [ ]:
fig, axes = plt.subplots(len(MODEL_SPECS), 2, figsize=(16, 4 * len(MODEL_SPECS)), sharex=False)
if len(MODEL_SPECS) == 1:
    axes = axes[None, :]

for row, (label, history) in enumerate(singular_history.items()):
    updates = history["updates"]
    for col, key in enumerate(["W1_singular", "W2_singular"]):
        singular_values = history[key]
        ax = axes[row, col]
        for rank in range(singular_values.shape[1]):
            ax.plot(updates, singular_values[:, rank], marker="o", markersize=2, label=f"rank {rank + 1}")
        ax.set_title(f"{label}: {key.replace('_', ' ')}")
        ax.set_xlabel("SGD update")
        ax.set_ylabel("singular value")
        ax.set_yscale("log")
        ax.set_xscale("log")
        ax.grid(True, alpha=0.3)
        ax.legend(ncol=2, fontsize=9)

plt.tight_layout()
plt.show()

## Data PCA Basis

All PCA plots use the covariance PCA basis of the training data, so both models are projected into the same visible-space coordinates.

In [ ]:
with torch.no_grad():
    centered_data = train_visible - train_visible.mean(dim=0, keepdim=True)
    cov_data = centered_data.t() @ centered_data / max(1, train_visible.shape[0] - 1)
    _, _, V_dataT = torch.linalg.svd(cov_data)

    data_proj = train_visible @ V_dataT.mT
    data_proj = data_proj.detach().cpu().numpy()

print("data_proj", data_proj.shape)

## Final Permanent Chains in PCA Space

The archive stores only the latest `parallel_chains`, not one copy per checkpoint. So this cell compares final permanent chains only.

In [ ]:
final_perm_proj = {}
final_perm_visible = {}

for label in MODEL_SPECS:
    final_update = int(saved_updates_by_model[label][-1])
    params, perm_chains, _ = load_model_with_activation(label, final_update)

    pc_proj = perm_chains["visible"] @ V_dataT.mT
    pc_proj = pc_proj.cpu().numpy()

    final_perm_visible[label] = perm_chains["visible"].detach().cpu()
    final_perm_proj[label] = pc_proj

    print(label, "update", final_update, "chains", tuple(perm_chains["visible"].shape))

In [ ]:
for label, pc_proj in final_perm_proj.items():
    plot_mult_PCA(
        data_proj[:, :8],
        pc_proj[:, :8],
        labels=["dataset", f"{label} permanent chains"],
    )

In [ ]:
for label, visible in final_perm_visible.items():
    plot_binary_image_grid(visible=visible, title=f"{label}: final permanent chains", shape=IMAGE_SHAPE, grid_size=(4, 4))

## Generated Samples Over Training

For selected checkpoints, we generate samples from the same initial Bernoulli noise. The kernel is DMALA because these are generic MLP binary EBMs.

In [ ]:
TRAINING_SAMPLE_NUM_CHECKPOINTS = 5
TRAINING_SAMPLE_NUM_CHAINS = 512
TRAINING_SAMPLE_STEPS = 10000
TRAINING_DMALA_ALPHA = 0.5

sample_updates_by_model = {
    label: selected_updates(saved_updates_by_model[label], TRAINING_SAMPLE_NUM_CHECKPOINTS)
    for label in MODEL_SPECS
}

training_samples = {}
training_sample_proj = {}

torch.manual_seed(0)
initial_visible = torch.bernoulli(torch.full((TRAINING_SAMPLE_NUM_CHAINS, num_visibles), 0.5, device=device, dtype=dtype))

for label in MODEL_SPECS:
    training_samples[label] = {}
    training_sample_proj[label] = {}

    for update in tqdm(sample_updates_by_model[label], desc=f"Samples over training: {label}"):
        params = load_params_with_activation(label, int(update))
        visible, acceptance = sample_visible(params=params, initial_visible=initial_visible, num_steps=TRAINING_SAMPLE_STEPS, alpha=TRAINING_DMALA_ALPHA)

        pc_proj = visible @ V_dataT.mT
        pc_proj = pc_proj.cpu().numpy()

        training_samples[label][int(update)] = visible.detach().cpu()
        training_sample_proj[label][int(update)] = pc_proj

        if acceptance is not None:
            print(label, int(update), "DMALA acceptance", acceptance)

In [ ]:
for label, proj_by_update in training_sample_proj.items():
    for update, pc_proj in proj_by_update.items():
        plot_mult_PCA(
            data_proj[:, :8],
            pc_proj[:, :8],
            labels=["dataset", f"{label} samples update {update}"],
        )

In [ ]:
SAMPLES_PER_CHECKPOINT = 8

for label, samples_by_update in training_samples.items():
    num_rows = len(samples_by_update)
    fig, axes = plt.subplots(
        num_rows,
        SAMPLES_PER_CHECKPOINT,
        figsize=(SAMPLES_PER_CHECKPOINT, 1.35 * num_rows),
        squeeze=False,
    )

    for row, (update, visible) in enumerate(samples_by_update.items()):
        for col in range(SAMPLES_PER_CHECKPOINT):
            ax = axes[row, col]
            ax.imshow(visible[col].view(*IMAGE_SHAPE), cmap="gray_r", vmin=0, vmax=1)
            ax.axis("off")

        axes[row, 0].set_ylabel(
            f"update {update}",
            rotation=0,
            labelpad=40,
            va="center",
            fontsize=10,
        )

    fig.suptitle(f"{label}: {SAMPLES_PER_CHECKPOINT} generated samples per checkpoint (dmala, {TRAINING_SAMPLE_STEPS} steps)")
    plt.tight_layout()
    plt.show()

## Mixing-Time Diagnostics

For each final model we run DMALA on the MLP visible energy. We compute the normalized spin autocorrelation and fit an exponential decay `rho(tau) ~= A exp(-tau / tau_exp)`.

In [ ]:
def spin_autocorr_fft(spins, max_lag, component_batch=4096, fft_device=None):
    if fft_device is None:
        fft_device = device

    num_times, num_chains, num_visibles = spins.shape
    max_lag = min(max_lag, num_times - 1)

    mean_spin = spins.to(torch.float32).mean(dim=(0, 1)).cpu()
    flat = spins.reshape(num_times, num_chains * num_visibles)
    n_fft = 1 << (2 * num_times - 1).bit_length()

    counts = torch.arange(num_times, num_times - max_lag - 1, -1, device=fft_device, dtype=torch.float32)

    numerator = torch.zeros(max_lag + 1, dtype=torch.float64)
    denominator = torch.tensor(0.0, dtype=torch.float64)

    for col_start in tqdm(range(0, flat.shape[1], component_batch), leave=False):
        col_stop = min(col_start + component_batch, flat.shape[1])
        block = flat[:, col_start:col_stop].to(device=fft_device, dtype=torch.float32)

        pixel_indices = torch.arange(col_start, col_stop, device="cpu") % num_visibles
        block = block - mean_spin[pixel_indices].to(fft_device).view(1, -1)

        block_fft = torch.fft.rfft(block, n=n_fft, dim=0)
        autocov = torch.fft.irfft(block_fft.abs().square(), n=n_fft, dim=0)[: max_lag + 1]
        autocov = autocov / counts[:, None]

        numerator += autocov.sum(dim=1).detach().cpu().double()
        denominator += autocov[0].sum().detach().cpu().double()

    return (numerator / denominator.clamp_min(1e-12)).numpy()


def collect_spin_trajectory(params, num_chains, num_steps, alpha=0.5):
    visible = torch.bernoulli(torch.full((num_chains, num_visibles), 0.5, device=device, dtype=dtype))
    chains = params.init_chains(num_samples=num_chains, start_v=visible)
    spins = torch.empty((num_steps + 1, num_chains, num_visibles), dtype=torch.int8, device="cpu")
    acceptances = []

    for step in tqdm(range(num_steps + 1), desc="trajectory dmala"):
        spins[step].copy_((2.0 * chains["visible"].detach().cpu() - 1.0).to(torch.int8))

        if step < num_steps:
            chains = params.sample_state(
                chains=chains,
                n_steps=1,
                kernel="dmala",
                kernel_params={"alpha": alpha},
            )
            acceptance = chains.get("acceptance")
            if torch.is_tensor(acceptance):
                acceptances.append(float(acceptance.detach().cpu()))

    mean_acceptance = float(np.mean(acceptances)) if acceptances else None
    return spins, mean_acceptance

In [ ]:
MIX_NUM_CHAINS = 256
MIX_NUM_STEPS = 30_000
MIX_MAX_LAG = 20_000
MIX_DMALA_ALPHA = 0.5

mixing_curves = {}
mixing_acceptance = {}

for label in MODEL_SPECS:
    final_update = int(saved_updates_by_model[label][-1])
    params = load_params_with_activation(label, final_update)

    spins, acceptance = collect_spin_trajectory(
        params=params,
        num_chains=MIX_NUM_CHAINS,
        num_steps=MIX_NUM_STEPS,
        alpha=MIX_DMALA_ALPHA,
    )
    mixing_curves[label] = spin_autocorr_fft(spins, max_lag=MIX_MAX_LAG, fft_device=device)
    mixing_acceptance[label] = acceptance
    if acceptance is not None:
        print(label, "dmala acceptance", acceptance)

In [ ]:
FIT_MIN_RHO = 0.05
FIT_MAX_RHO = 0.6
mixing_tau = {}

fig, axes = plt.subplots(len(MODEL_SPECS), 1, figsize=(6, 4 * len(MODEL_SPECS)), sharex=True, sharey=True)
if len(MODEL_SPECS) == 1:
    axes = [axes]

for ax, (label, rho) in zip(axes, mixing_curves.items()):
    lags = np.arange(len(rho))
    rho_positive = np.clip(rho, 1e-12, None)

    ax.plot(lags, rho_positive, label="autocorr")

    fit_mask = (lags > 0) & np.isfinite(rho) & (rho > FIT_MIN_RHO) & (rho < FIT_MAX_RHO)

    if fit_mask.sum() >= 2:
        slope, intercept = np.polyfit(lags[fit_mask], np.log(rho[fit_mask]), deg=1)
        tau_exp = -1.0 / slope if slope < 0 else np.inf
        fit_lags = lags[fit_mask]
        fit_curve = np.exp(intercept + slope * fit_lags)
        ax.plot(fit_lags, fit_curve, "--", label=f"exp fit, tau={tau_exp:.1f}")
    else:
        tau_exp = np.nan

    mixing_tau[label] = tau_exp
    acceptance = mixing_acceptance[label]
    accept_text = "" if acceptance is None else f", acc={acceptance:.3f}"

    ax.set_title(f"{label} | dmala{accept_text}")
    ax.set_xlabel("lag")
    ax.set_ylabel("rho")
    ax.set_yscale("log")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

for label, tau_exp in mixing_tau.items():
    print(f"{label}: dmala tau = {tau_exp:.2f} steps")